In [ ]:
import torch

In [2]:
data = torch.load('data_tensor_2.pt')

In [40]:
data[0,:,2]

tensor([1., 0.])

In [41]:
data[0]

tensor([[0., 0., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.,
         2., 0., 0., 0., 0.],
        [1., 1., 0., 2., 0., 0., 0., 0., 0., 0., 0., 0., 2., 1., 0., 0., 0., 0.,
         0., 0., 0., 0., 0.]])

In [3]:
data.shape

torch.Size([1749370, 2, 23])

In [4]:
import torch
import numpy as np

# --- SETUP (DO THIS ONLY ONCE) ---
# Assume 'data_numpy' is your data in a NumPy array.
# For demonstration, let's create some dummy data.
# In your code, 'data' would have shape (num_samples, 23)
# and 'w' would have shape (23, 2) for binary classification.
num_samples = 10000
num_features = 23
# 1. Determine the best available device (GPU or CPU)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

# 2. Load data and move it to the target device ONCE.
#    This is the most critical optimization.
data = data.to(device)
# --- END OF SETUP ---


def get_fitness(w: np.ndarray) -> float:
    """
    Optimized fitness function.
    Assumes 'data' and 'device' are defined in the outer scope.
    """
    # 1. Convert numpy array 'w' to a tensor directly on the target device.
    #    This minimizes CPU-GPU data transfer.
    w_tensor = torch.tensor(w, dtype=torch.float32, device=device)

    # 2. Disable gradient calculation for speed and memory efficiency.
    with torch.no_grad():
        # 3. Perform the matrix multiplication (most expensive step).
        #    This now happens entirely on the target device.
        U = torch.matmul(data, w_tensor)

    # 4. Calculate accuracy efficiently (unused variables are removed).
    #    This is a binary classification accuracy for class 0.
    #    The result of the comparison is a boolean tensor, also on the device.
    correct = (U[:, 0] > U[:, 1])

    # 5. Calculate the mean and move the final scalar result to the CPU.
    accuracy = correct.float().mean()
    return accuracy.item()

Using device: mps


In [18]:
import numpy as np
from sklearn.cluster import DBSCAN
from scipy.spatial.distance import cdist

def select_diverse_points_greedy(points, fitnesses, k):
    """
    Selects k diverse points using a greedy farthest-point sampling strategy.

    Args:
        points (np.ndarray): Array of candidate points (shape: [N, D]).
        fitnesses (np.ndarray): Array of fitness scores for each point (shape: [N]).
        k (int): The number of diverse points to select.

    Returns:
        np.ndarray: The k selected diverse points.
    """
    if len(points) <= k:
        return points

    # Sort points by fitness in descending order
    sorted_indices = np.argsort(fitnesses)[::-1]
    sorted_points = points[sorted_indices]

    selected_points = [sorted_points[0]]
    candidate_points = sorted_points[1:]

    for _ in range(k - 1):
        if len(candidate_points) == 0:
            break
            
        # Calculate the minimum distance from each candidate to any already selected point
        # cdist calculates distances between each point in candidate_points and selected_points
        distances = cdist(candidate_points, selected_points)
        min_distances = np.min(distances, axis=1)

        # Select the candidate that is farthest from any selected point
        farthest_idx = np.argmax(min_distances)
        
        selected_points.append(candidate_points[farthest_idx])
        
        # Remove the selected point from the candidates for the next iteration
        candidate_points = np.delete(candidate_points, farthest_idx, axis=0)
        
    return np.array(selected_points)

In [ ]:
import numpy as np
import cma
from scipy.stats import qmc
from scipy.spatial.distance import pdist, squareform
from tqdm import tqdm


def find_peaks(
    dimensions=23,
    bounds=(-10, 10),
    num_samples=100000,
    fitness_threshold=0.6,
    patience_iterations=100,
    patience_tolerance=1e-5,
    sensitivity=0.0001,
    top_k_candidates=250,
):
    """
    Calculates the peaks of a landscape function using CMA-ES for local search.

    Args:
        dimensions (int): The number of dimensions of the input vector.
        bounds (tuple): A tuple containing the lower and upper bounds for each dimension.
        num_samples (int): The number of initial samples to generate using Sobol sequences.
        fitness_threshold (float): The minimum fitness score for a sample to undergo local search.
        patience_iterations (int): The number of CMA-ES generations to wait for improvement before stopping.
        patience_tolerance (float): The minimum fitness improvement to reset the patience counter.
        sensitivity (float): The distance within which to consider peaks as identical.

    Returns:
        np.ndarray: An array of the identified peak vectors.
    """
    lower_bound, upper_bound = bounds

    # 1. Generate initial samples using Sobol sequences.
    print("Step 1: Generating initial samples...")
    sampler = qmc.Sobol(d=dimensions)
    samples = sampler.random(n=num_samples)
    scaled_samples = qmc.scale(samples, lower_bound, upper_bound)
    print("Done.")

    # 2. Filter samples based on the fitness threshold.
    print("\nStep 2: Evaluating fitness of initial samples...")
    initial_fitness = np.array(
        [get_fitness(s) for s in tqdm(scaled_samples, desc="Evaluating initial samples")]
    )
    promising_starts = scaled_samples[initial_fitness > fitness_threshold]

    if len(promising_starts) == 0:
        print("\nNo starting points found above the fitness threshold. Consider lowering the threshold.")
        return np.array([])

    print(f"Found {len(promising_starts)} promising starting points.")

        # 2a. Select a diverse subset of k points for expensive optimization
    if top_k_candidates is not None and len(promising_starts) > top_k_candidates:
        print(f"\nStep 2a: Selecting top {top_k_candidates} diverse candidates from {len(promising_starts)} promising points...")
        
        # We need the fitnesses of the promising points, which we already calculated
        promising_fitnesses = initial_fitness[initial_fitness > fitness_threshold]

        # Use the greedy method (recommended for simplicity)
        cma_seeds = select_diverse_points_greedy(promising_starts, promising_fitnesses, top_k_candidates)
    

        print(f"Selected {len(cma_seeds)} points to seed CMA-ES.")
    else:
        cma_seeds = promising_starts
    promising_starts = cma_seeds

    # 3. Perform local search with CMA-ES from each promising start point.
    print("\nStep 3: Optimizing with CMA-ES...")
    potential_peaks = []
    
    # CMA-ES minimizes, so we need to find the minimum of the negative fitness.
    def objective_function(x):
        return -get_fitness(x)

    # Set up CMA-ES options based on user's patience criteria.
    # 'verbose': -9 silences the output from each CMA-ES run.
    cma_options = {
        'bounds': [lower_bound, upper_bound],
        'tolfun': patience_tolerance,
        'tolstagnation': patience_iterations,
        'verbose': -9,
    }

    for start_point in tqdm(promising_starts, desc="Running CMA-ES"):
        # Initial standard deviation (step size). A reasonable guess is 1/4 of the search range.
        sigma0 = (upper_bound - lower_bound) / 4.0
        
        # Instantiate and run the CMA-ES algorithm.
        es = cma.CMAEvolutionStrategy(start_point, sigma0, cma_options)
        es.optimize(objective_function)
        
        # The result 'es.result.xbest' is the vector of the found optimum.
        potential_peaks.append(es.result.xbest)

    if not potential_peaks:
        print("CMA-ES did not converge on any peaks.")
        return np.array([])

    potential_peaks = np.array(potential_peaks)
    print(f"Identified {len(potential_peaks)} potential peaks after CMA-ES.")

    # 4. Filter and cluster the identified peaks to remove duplicates.
    print("\nStep 4: Clustering peaks to find unique solutions...")
    distances = pdist(potential_peaks)
    distance_matrix = squareform(distances)

    unique_peaks = []
    is_peak_processed = [False] * len(potential_peaks)

    for i in tqdm(range(len(potential_peaks)), desc="Clustering Peaks"):
        if not is_peak_processed[i]:
            close_indices = np.where(distance_matrix[i] < sensitivity)[0]
            cluster = potential_peaks[close_indices]
            
            # Represent the cluster by the member with the highest fitness.
            cluster_fitness = [get_fitness(p) for p in cluster]
            best_in_cluster = cluster[np.argmax(cluster_fitness)]
            unique_peaks.append(best_in_cluster)

            for idx in close_indices:
                is_peak_processed[idx] = True

    final_peaks = np.array(unique_peaks)
    print(f"Found {len(final_peaks)} unique peaks after clustering.")
    return final_peaks

In [20]:
peaks = find_peaks()

/var/folders/pw/hbp4440d5m3gzjnln2nycrc80000gn/T/ipykernel_66460/3091779359.py:38: UserWarning: The balance properties of Sobol' points require n to be a power of 2.
  samples = sampler.random(n=num_samples)


Step 1: Generating initial samples...
Done.

Step 2: Evaluating fitness of initial samples...


Evaluating initial samples: 100%|██████████| 100000/100000 [19:56<00:00, 83.58it/s] 


Found 7144 promising starting points.

Step 2a: Selecting top 250 diverse candidates from 7144 promising points...
Selected 250 points to seed CMA-ES.

Step 3: Optimizing with CMA-ES...


Running CMA-ES: 100%|██████████| 250/250 [4:01:54<00:00, 58.06s/it]  


Identified 250 potential peaks after CMA-ES.

Step 4: Clustering peaks to find unique solutions...


Clustering Peaks: 100%|██████████| 250/250 [00:02<00:00, 84.13it/s]


Found 250 unique peaks after clustering.


In [22]:
# 1. Calculate individual accuracies for each jury member
print("=== Individual Jury Member Accuracies ===")
individual_accuracies = []

for i, member_weights in enumerate(peaks):
    accuracy = get_fitness(member_weights)
    individual_accuracies.append(accuracy)
    print(f"Jury Member {i+1}: {accuracy:.6f}")

print(f"\nAverage individual accuracy: {np.mean(individual_accuracies):.6f}")
print(f"Best individual accuracy: {np.max(individual_accuracies):.6f}")
print(f"Worst individual accuracy: {np.min(individual_accuracies):.6f}")
print(f"Standard deviation: {np.std(individual_accuracies):.6f}")

=== Individual Jury Member Accuracies ===
Jury Member 1: 0.787718
Jury Member 2: 0.787702
Jury Member 3: 0.787678
Jury Member 4: 0.787695
Jury Member 5: 0.787641
Jury Member 6: 0.787705
Jury Member 7: 0.787617
Jury Member 8: 0.787666
Jury Member 9: 0.787689
Jury Member 10: 0.787666
Jury Member 11: 0.787650
Jury Member 12: 0.787694
Jury Member 13: 0.787693
Jury Member 14: 0.787695
Jury Member 15: 0.787741
Jury Member 16: 0.787718
Jury Member 17: 0.787690
Jury Member 18: 0.787720
Jury Member 19: 0.787676
Jury Member 20: 0.787666
Jury Member 21: 0.787673
Jury Member 22: 0.787708
Jury Member 23: 0.787636
Jury Member 24: 0.787664
Jury Member 25: 0.787672
Jury Member 26: 0.787695
Jury Member 27: 0.787658
Jury Member 28: 0.787675
Jury Member 29: 0.787686
Jury Member 30: 0.787714
Jury Member 31: 0.787741
Jury Member 32: 0.787641
Jury Member 33: 0.787702
Jury Member 34: 0.787677
Jury Member 35: 0.787674
Jury Member 36: 0.787711
Jury Member 37: 0.787668
Jury Member 38: 0.787676
Jury Member 39: 0

In [ ]:
def calculate_jury_accuracy(jury, data_tensor):
    """
    Calculates the accuracy of a jury of weight vectors using majority voting.
    """
    device = "cpu"
    jury = torch.tensor(jury, dtype=torch.float32, device=device)
    jury.to(device)
    data_tensor.to(device)
    n_samples = data_tensor.shape[0]
    all_votes = []

    # Get votes from each jury member
    for i, member_weights in enumerate(jury):
        print(f"Collecting votes from Jury Member {i+1}/{len(jury)}...")
        if type(member_weights) is not torch.Tensor:
            member_weights = torch.tensor(member_weights, dtype=torch.float32)
        
        U = torch.matmul(data_tensor, member_weights)
        # A vote for 0 means U[:, 0] > U[:, 1], a vote for 1 means the opposite
        votes = (U[:, 1] > U[:, 0]).long()
        all_votes.append(votes)
    
    # Stack votes for easy majority calculation
    votes_tensor = torch.stack(all_votes)
    
    # Calculate majority vote for each sample
    # A majority vote of 0 means most jury members preferred the first option.
    # A majority vote of 1 means most jury members preferred the second option.
    majority_votes, _ = torch.mode(votes_tensor, dim=0)
    
    # Ground truth is always 0 (the first option is better)
    y_true = torch.zeros(n_samples, dtype=torch.long)
    
    # Calculate final accuracy
    correct_predictions = (majority_votes == y_true).sum().item()
    accuracy = correct_predictions / n_samples
    
    return accuracy




In [36]:
device = "cpu"
data = data.to(device)
acc = calculate_jury_accuracy(peaks, data)

In [37]:
acc

0.7877413011541298